# String Output Parser

Output Parser — processes what LLM already returned
pythonchain = llm | StrOutputParser()

Doesn't tell LLM what shape to return
Just cleans/extracts from whatever LLM returns
Best when you just need plain text or simple extraction


Simple analogy:

Structured output = telling a chef to plate food a specific way
Output parser = taking whatever food comes out and putting it on your own plate

In [2]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()

llm=ChatOpenAI(model='gpt-4o-mini',temperature=0.6)

prompt_template=PromptTemplate.from_template('what is the definition of {word}')
prompt=prompt_template.invoke({'word':'langchain'})

response=llm.invoke(prompt)
print(response.content)


LangChain is a framework designed to facilitate the development of applications that utilize large language models (LLMs). It provides a modular and flexible approach to building applications by integrating various components such as prompt management, memory, and chains of calls to LLMs and other data sources.

Key features of LangChain include:

1. **Chains**: Sequences of calls that can include LLM invocations, API calls, or other functions, allowing developers to create complex workflows.

2. **Prompts**: Tools for managing and optimizing the prompts sent to LLMs, which can significantly affect the quality of the model's output.

3. **Memory**: Capabilities to maintain state across interactions, enabling applications to remember previous inputs and outputs, thus providing a more coherent user experience.

4. **Agents**: Components that can make decisions about which actions to take based on user input, often employing LLMs to determine the best course of action.

5. **Integrations*

# Using parsers

In [9]:
output_parser=StrOutputParser()
template=PromptTemplate.from_template('what is the definition of {word}')
prompt=template.invoke({'word':'langchain'})
result=llm.invoke(prompt)
parsed_output=output_parser.invoke(result)
print(parsed_output)

LangChain is a framework designed for developing applications that utilize large language models (LLMs). It provides tools and components that facilitate the integration of LLMs with various data sources, APIs, and other services, enabling developers to create more complex and capable applications.

Key features of LangChain include:

1. **Modularity**: LangChain offers a modular architecture, allowing developers to mix and match different components, such as prompts, models, and memory management systems, to suit their specific application needs.

2. **Chains**: The framework allows for the creation of "chains," which are sequences of operations that can include calling an LLM, processing its output, and interacting with other systems or data sources.

3. **Agents**: LangChain can create agents that can make decisions based on user input, the context of a conversation, or external data, enabling more dynamic and interactive applications.

4. **Data Connection**: It supports various in

## with chain 

In [5]:
template=PromptTemplate.from_template('what is the definition of {word}')
llm=ChatOpenAI(model='gpt-4o-mini',temperature=0.6)
parser=StrOutputParser()

chain= template | llm | parser 
result=chain.invoke({'word':'langchain'})
print(result)

LangChain is a framework designed for developing applications that utilize language models, particularly those based on transformer architectures like GPT (Generative Pre-trained Transformer). It provides a structured way to integrate various components, such as natural language processing (NLP) models, data sources, and application logic, to create more sophisticated and context-aware applications.

LangChain facilitates the following:

1. **Chain of Thought**: It allows developers to create workflows that can combine multiple calls to language models, enabling more complex reasoning and interaction patterns.
2. **Data Integration**: LangChain can connect to various data sources, making it easier to pull in context or information that the language model can use to generate more relevant responses.
3. **Tooling and Utilities**: The framework often includes utilities for prompt management, memory handling, and API integrations, which streamline the development process.
4. **Agent Capabi

# JSON OutParser

In [13]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

json_parser=JsonOutputParser()
template=PromptTemplate.from_template(""" 
Extact the following and pass it in json format:
- Name
-Price
-Category
-Features (list)

                                      
test:{text}
format_instructions:{format_instructions}""",

partial_variables={'format_instructions':json_parser.get_format_instructions()})

chain = template | llm | json_parser

text=""" samsugng galaxy s23 ultra. what is the price and camera. can you tell me the price in india"""

result = chain.invoke({'text':text})
print(result)


{'Name': 'Samsung Galaxy S23 Ultra', 'Price': 'Price in India not specified', 'Category': 'Smartphone', 'Features': ['High-resolution camera', 'Large display', 'Long battery life', 'S Pen support', '5G connectivity']}


# Pydantic output parser

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel,Field 
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from typing import List,Optional



class p(BaseModel):
  title: str = Field(..., description="Name of the movie") # ... means a requred field
  release_year: int = Field(..., description= "Year the movie was released")
  genres: List[str] = Field(..., description= "List of genres the movie belongs to")
  rating: float = Field(None, description= "Average rating of the movie on a scale of 1 to 10")
  box_office: Optional[float] = Field( None , description= "Worldwide box office in millions USD, if known")


pydantic_parser=PydanticOutputParser(pydantic_object=Movie)
template= PromptTemplate.from_template("""Extract information from the following text and your knowledge:
{text}
{format_instructions}""",
partial_variables={'format_instructions':pydantic_parser.get_format_instructions()})

chain= template | llm | pydantic_parser
text = "Inception is a 2010 sci-fi thriller directed by Christopher Nolan with a rating of 8.8 and box office of 836 million"
result=chain.invoke({'text':text})
print(result)


title='Inception' release_year=2010 genres=['sci-fi', 'thriller'] rating=8.8 box_office=836.0
